In [2]:
import pandas as pd 
import numpy as np
import joblib
import time
import sklearn.metrics
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier

## Load dataset generated by CovaS

In [3]:
print("Loading train...")
train_df = pd.read_csv('./DS/augmented88.csv', low_memory=False)
print("Loading test...")
test_df= pd.read_csv('./DS/iec104_test.csv', low_memory=False)
print(train_df.shape, test_df.shape)
FS = ['flow total IEC104_U_Message packets', 'bw packet APDU length max', 'cot=1', 'type_id_process_information_in_monitor_direction', 'total flow packets', 'type_id_process_information_in_control_direction', 'bw packet APDU length std', 'flow packets APDU total length', 'type_id_system_information_in_control_direction', 'bw total IEC104_U_Message packets', 'flow packet APDU length std', 'bw total IEC104_S_Message packets', 'flow down/up ratio', 'cot=3', 'flow iec104 packts/s', 'flow packet APDU length max', 'flow packet APDU length mean', 'flow IAT min', 'bw IAT min', 'bw packets APDU total length', 'flow IAT tot', 'flow IAT max', 'fw packet APDU length std', 'bw iec104 packts/s', 'init fw window bytes', 'flow total IEC104_I_Message_SingleIOA packets', 'flow active time std', 'flow idle time std', 'bw packet APDU length mean', 'flow total IEC104_S_Message packets', 'fw iec104 packts/s', 'fw TCP total header length', 'flow IAT std', 'fw IAT max', 'fw IAT std', 'fw_subflow_bytes', 'bw IAT std', 'bw_subflow_packets', 'flow idle time mean', 'fw total IEC104_I_Message_SingleIOA packets', 'flow active time min', 'fw iec104 bytes/s', 'cot=10', 'flow idle time max', 'bw iec104 bytes/s', 'bw IAT max', 'flow active time mean', 'flow IAT mean', 'flow active time max', 'total fw packets', 'fw iAT tot', 'bw IAT tot', 'flow idle time min', 'flow iec104 bytes/s', 'bw IAT mean', 'fw IAT min', 'fw IAT mean', 'bw avg bytes/bulk', 'init bw window bytes', 'bw avg packets/bulk', 'fw total IEC104_U_Message packets', 'bw_subflow_bytes']

# X_train = train_df.drop(['Label'], axis=1)
X_train = train_df[FS]
y_train = train_df['Label']
# X_test = test_df.drop(['Label'], axis=1)
X_test = test_df[FS]
y_test = test_df['Label']

Loading train...
Loading test...
(11200, 63) (2028, 112)


## XGBoost

In [4]:
xgb_params = {
    'tree_method': 'gpu_hist',         
    'predictor': 'gpu_predictor',  
    'max_depth': 64,# 128,
    'n_estimators': 500, #5000,
    'learning_rate': 0.1, # best 0.5;  0.01 0.05
    'eval_metric': 'auc',
    'booster': 'gbtree',
    'objective':"multi:softprob",     
    'random_state': 42,
    'early_stopping_rounds': 20,
}


print("XGBClassifier Starting")
xgb_model = XGBClassifier(**xgb_params)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred = xgb_model.predict(X_test)
print("XGBClassifier Finished")

print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

XGBClassifier Starting
XGBClassifier Finished
Accuracy: 0.878698224852071
Precision: 0.8774069142282399
DR: 0.878698224852071
F1: 0.8752134049573529
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      1    168      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     32      0      1      1      0     86     28      7     13      0
  8      0      0      0      0      0      0      0     3

In [10]:
joblib.dump(xgb_model, './models/aug_xgb.pkl')

['./models/aug_xgb.pkl']

## ExtraTree

In [11]:
et_params = {
    "n_estimators": 85,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

print("ExtraTreesClassifier Starting")
et_model = ExtraTreesClassifier(**et_params)
et_model.fit(X=X_train, y=y_train)
et_start_time = time.time()
y_pred = et_model.predict(X_test)
et_end_time = time.time()
et_time = et_end_time - et_start_time
print("ExtraTreesClassifier Finished")
print("ExtraTrees Time:", et_end_time - et_start_time)
print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

ExtraTreesClassifier Starting
ExtraTreesClassifier Finished
ExtraTrees Time: 0.08540678024291992
Accuracy: 0.8698224852071006
Precision: 0.8698218074289232
DR: 0.8698224852071007
F1: 0.86705646601402
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      0    168      1      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     23      0      0      0      0     96     26     10     13      0
  8    

In [10]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn import metrics as sklearn

# === Hàm phụ (nếu đã có thì bỏ qua) ===
def calculate_macro_tpr_fpr(cm):
    C = cm.shape[0]
    tprs, fprs = [], []
    for c in range(C):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
        tprs.append(tpr)
        fprs.append(fpr)
    return float(np.mean(tprs)), float(np.mean(fprs))

# === Tham số cố định ===
et_params = {
    "n_estimators": 70,        # sẽ thay trong vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

START, END, STEP = 60, 90, 1  # có thể đổi STEP=1 để quét mịn

print("ExtraTreesClassifier Sweep Starting")

records = []
best_key = None
best_model, best_cm, best_pred = None, None, None
best_n, best_time = None, None

for n in range(START, END + 1, STEP):
    params = et_params.copy()
    params["n_estimators"] = n

    model = ExtraTreesClassifier(**params)
    model.fit(X_train, y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_test, pred)
    precision = sklearn.precision_score(y_test, pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_test, pred, average='macro')
    recall = sklearn.recall_score(y_test, pred, average='macro')
    cm = sklearn.confusion_matrix(y_test, pred)
    et_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": et_fp
    })

    key = (acc, f1, recall, -n)  # tie-break
    if best_key is None or key > best_key:
        best_key = key
        best_model, best_cm, best_pred = model, cm, pred
        best_n, best_time = n, t1 - t0

print("ExtraTreesClassifier Sweep Finished")

# === Tổng hợp kết quả ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], 
    ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === Report tốt nhất ===
et_acc = sklearn.accuracy_score(y_test, best_pred)
et_precision = sklearn.precision_score(y_test, best_pred, average='macro', zero_division=0)
et_f1 = sklearn.f1_score(y_test, best_pred, average='macro')
et_recall = sklearn.recall_score(y_test, best_pred, average='macro')
et_cm = best_cm
et_fp = et_cm[0, 1] if et_cm.shape == (2, 2) else None
et_tpr, et_fpr = calculate_macro_tpr_fpr(et_cm)

print("\n===== Best ExtraTrees report =====")
print(f"Best n_estimators: {best_n}")
print("ExtraTrees Time:", best_time)
print("ExtraTrees Accuracy:", et_acc)
print("ExtraTrees Precision:", et_precision)
print("ExtraTrees F1:", et_f1)
print("ExtraTrees Recall:", et_recall)
print("ExtraTrees FP:", et_fp)
print("ExtraTrees CM:\n", et_cm)
print(f'ExtraTrees Macro-average TPR: {et_tpr}')
print(f'ExtraTrees Macro-average FPR: {et_fpr}')

# df_results.to_csv("./et_sweep_n_estimators_30_500.csv", index=False)


ExtraTreesClassifier Sweep Starting
ExtraTreesClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0            85  0.863412  0.859691      0.863412  0.084643
1            90  0.862919  0.859154      0.862919  0.109720
2            83  0.862426  0.858736      0.862426  0.079463
3            86  0.862426  0.858630      0.862426  0.088089
4            87  0.862426  0.858580      0.862426  0.103691
5            84  0.861933  0.858125      0.861933  0.090223
6            88  0.861933  0.858086      0.861933  0.109535
7            89  0.861933  0.857972      0.861933  0.105265
8            75  0.860947  0.856953      0.860947  0.070961
9            81  0.860947  0.856883      0.860947  0.082691

===== Best ExtraTrees report =====
Best n_estimators: 85
ExtraTrees Time: 0.08464336395263672
ExtraTrees Accuracy: 0.8634122287968442
ExtraTrees Precision: 0.8628522548640115
ExtraTrees F1: 0.8596908602856262
ExtraTrees Recall: 

In [12]:
joblib.dump(et_model, './models/aug_et.pkl')

['./models/aug_et.pkl']

## RandomForest

In [13]:
rf_params = {
    "n_estimators": 50,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

print("RandomForestClassifier Starting")
rf_model = RandomForestClassifier(**rf_params)
rf_model.fit(X=X_train, y=y_train)
rf_start_time = time.time()
y_pred = rf_model.predict(X_test)
rf_end_time = time.time()
rf_time = rf_end_time - rf_start_time
print("RandomForestClassifier Finished")
print("RandomForest Time:", rf_end_time - rf_start_time)
print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

RandomForestClassifier Starting
RandomForestClassifier Finished
RandomForest Time: 0.049573659896850586
Accuracy: 0.8673570019723866
Precision: 0.8674696089049899
DR: 0.8673570019723865
F1: 0.8638994040958307
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      1    168      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      0    169      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     33      0      0      0      0     90     23      8     14      

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# === Hàm phụ (nếu bạn đã có thì bỏ qua) ===
def calculate_macro_tpr_fpr(cm):
    """
    cm: confusion_matrix (C x C)
    Trả về (macro_TPR, macro_FPR)
    """
    C = cm.shape[0]
    tprs, fprs = [], []
    for c in range(C):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
        tprs.append(tpr)
        fprs.append(fpr)
    return float(np.mean(tprs)), float(np.mean(fprs))

# === Tham số cố định ===
rf_params = {
    "n_estimators": 60,        # sẽ bị thay mỗi vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

START, END, STEP = 50, 90, 1   # có thể đổi STEP=1 nếu muốn quét mịn hơn

print("RandomForestClassifier Sweep Starting")

records = []
best_key = None  # (acc, f1, recall, -n_estimators) để có tie-break rõ ràng
best_model = None
best_cm = None
best_pred = None
best_n = None
best_time = None

for n in range(START, END + 1, STEP):
    params = rf_params.copy()
    params["n_estimators"] = n

    model = RandomForestClassifier(**params)
    model.fit(X=X_train, y=y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_true=y_test, y_pred=pred)
    precision = sklearn.precision_score(y_true=y_test, y_pred=pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_true=y_test, y_pred=pred, average='macro')
    recall = sklearn.recall_score(y_true=y_test, y_pred=pred, average='macro')
    cm = sklearn.confusion_matrix(y_true=y_test, y_pred=pred)
    # Lưu ý: rf_fp = cm[0, 1] chỉ có ý nghĩa khi bài toán nhị phân và label 0/1 nằm đúng trục
    rf_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": rf_fp
    })

    # Tie-break: ưu tiên acc -> f1 -> recall; nếu vẫn hòa, chọn n_estimators nhỏ hơn
    key = (acc, f1, recall, -n)
    if (best_key is None) or (key > best_key):
        best_key = key
        best_model = model
        best_cm = cm
        best_pred = pred
        best_n = n
        best_time = t1 - t0

print("RandomForestClassifier Sweep Finished")

# === Tổng hợp kết quả thành DataFrame và in Top-10 theo Accuracy ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === In report mô hình tốt nhất (giữ format bạn đang dùng) ===
rf_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=best_pred)
rf_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=best_pred, average='macro', zero_division=0)
rf_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_cm = best_cm
rf_fp = rf_cm[0, 1] if rf_cm.shape == (2, 2) else None
rf_tpr, rf_fpr = calculate_macro_tpr_fpr(rf_cm)

print("\n===== Best RandomForest report =====")
print(f"Best n_estimators: {best_n}")
print("RandomForest Time:", best_time)
print("RandomForest Accuracy:", rf_acc)
print("RandomForest Precision:", rf_precision)
print("RandomForest F1:", rf_f1)
print("RandomForest Recall:", rf_recall)
print("RandomForest FP:", rf_fp)
print("RandomForest CM:", rf_cm)
print(f'RandomForest Macro-average TPR: {rf_tpr}')
print(f'RandomForest Macro-average FPR: {rf_fpr}')

RandomForestClassifier Sweep Starting
RandomForestClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0            50  0.865385  0.861248      0.865385  0.063664
1            57  0.864892  0.860603      0.864892  0.054941
2            61  0.864398  0.860094      0.864398  0.066899
3            60  0.864398  0.860092      0.864398  0.199380
4            62  0.864398  0.859927      0.864398  0.072500
5            51  0.863905  0.859825      0.863905  0.055553
6            55  0.863905  0.859774      0.863905  0.062425
7            79  0.863905  0.859651      0.863905  0.211811
8            56  0.863905  0.859538      0.863905  0.070011
9            52  0.863412  0.859280      0.863412  0.059841

===== Best RandomForest report =====
Best n_estimators: 50
RandomForest Time: 0.06366395950317383
RandomForest Accuracy: 0.8653846153846154
RandomForest Precision: 0.8659122731904888
RandomForest F1: 0.861248271298665
Random

In [14]:
joblib.dump(rf_model, './models/aug_rf.pkl')

['./models/aug_rf.pkl']

## LGBM

In [7]:
from lightgbm import LGBMClassifier, early_stopping

lgbm_params = {
 "boosting_type": "gbdt",
    "objective": "multiclass",
    "num_class": len(y_train.unique()),
    "metric": "multi_logloss",

    "learning_rate": 0.1,
    "n_estimators": 5000,
    "num_leaves": 192,
    "max_depth": -1,

    "min_child_samples": 48,
    "reg_alpha": 0.15,
    "reg_lambda": 0.8,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,

    "device": "cpu",
    "verbosity": -1,
    "random_state": 42,
    "n_jobs": -1, 
}

# Train LightGBM Model
print("LightGBM Starting")
lgbm_model = LGBMClassifier(**lgbm_params)
lgbm_model.fit(
    X=X_train, y=y_train, 
    eval_set=[(X_test, y_test)],
    callbacks=[early_stopping(stopping_rounds=30)],
)
y_pred = lgbm_model.predict(X_test)
print("LightGBM Finished")
print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

LightGBM Starting
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[76]	valid_0's multi_logloss: 0.358902
LightGBM Finished
Accuracy: 0.8717948717948718
Precision: 0.8695218326692635
DR: 0.8717948717948718
F1: 0.8683020552017893
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      1    168      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1  

In [15]:
joblib.dump(lgbm_model, './models/aug_lgbm.pkl')

['./models/aug_lgbm.pkl']

## catboost

In [8]:
from catboost import CatBoostClassifier

cat_params = {
    "iterations": 5000,
    "depth": 15,
    "learning_rate": 0.1,
    "loss_function": "MultiClass",
    "task_type": "GPU",
    "random_seed": 42,
    "eval_metric": "TotalF1",      # macro F1 cho đa lớp
    "od_type": "Iter",             # early stopping kiểu số vòng
    "od_wait": 30,                 # tương đương early_stopping_rounds=30
    "classes_count": len(y_train.unique())
}

print("CatBoost Starting")
cat_model = CatBoostClassifier(**cat_params)

# fit model
cat_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=False
)

cat_start_time = time.time()
y_pred = cat_model.predict(X_test)
cat_end_time = time.time()
cat_time = cat_end_time - cat_start_time
print("CatBoost Finished")
print("CatBoost Time:", cat_time)
print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

CatBoost Starting
CatBoost Finished
CatBoost Time: 0.012968301773071289
Accuracy: 0.8841222879684418
Precision: 0.8824940327578643
DR: 0.8841222879684419
F1: 0.8823782869208734
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      1    168      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      3    164      0      2      0      0      0      0      0
  5      0      0      0      0      0    169      0      0      0      0      0      0
  6      0      0      0      2      0      0    167      0      0      0      0      0
  7      0      0     19      2      1      0      3    108     26      5      5      0
  8      0      0      0      

In [16]:
joblib.dump(cat_model, './models/aug_cat.pkl')

['./models/aug_cat.pkl']

## biLSTM

In [30]:
# ============================
# LSTM cho Tabular — IEC/ICS preset — KHÔNG focus lớp nào
#  - Stratified split (VAL 15%)
#  - StandardScaler
#  - DataLoader (shuffle, không sampler/boost)
#  - LSTM (2 tầng, BiLSTM) + AttnPooling
#  - Mixup công bằng cho mọi lớp
#  - EMA (float-only) device-safe
#  - Early-stopping theo Macro-F1 (VAL)
#  - Đánh giá: Acc, Macro-F1, per-class metrics
# ============================
import math, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, precision_recall_fscore_support

# ----- Device & seed -----
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); np.random.seed(42)

# ===== Dữ liệu =====
X_full = train_df[FS].values
y_full = train_df['Label'].values
X_test = test_df[FS].values
y_test = test_df["Label"].values

num_classes  = int(max(y_full.max(), y_test.max())) + 1
num_features = X_full.shape[1]

# ===== 1) Stratified split: train_in / val (15%) =====
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss.split(X_full, y_full))
X_tr_in, y_tr_in = X_full[train_idx], y_full[train_idx]
X_val,   y_val   = X_full[val_idx],   y_full[val_idx]

# ===== 2) Scaler =====
scaler = StandardScaler()
X_tr_in_sc = scaler.fit_transform(X_tr_in)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# ===== 3) DataLoaders =====
Xtr_t = torch.tensor(X_tr_in_sc, dtype=torch.float32)
ytr_t = torch.tensor(y_tr_in,    dtype=torch.long)
Xva_t = torch.tensor(X_val_sc,   dtype=torch.float32)
yva_t = torch.tensor(y_val,      dtype=torch.long)
Xte_t = torch.tensor(X_test_sc,  dtype=torch.float32)
yte_t = torch.tensor(y_test,     dtype=torch.long)

batch_size = 2048
train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(TensorDataset(Xva_t, yva_t), batch_size=4096, shuffle=False, pin_memory=True)
test_loader  = DataLoader(TensorDataset(Xte_t, yte_t), batch_size=4096, shuffle=False, pin_memory=True)

# ===== 4) Tiện ích: chia feature thành "chuỗi" =====
# Ta coi features như một chuỗi gồm S steps, mỗi step có input_size = STEP_DIM
# (pad 0 nếu F không chia hết)
STEP_DIM = 16  # bạn có thể thử 8/16/32
def as_sequence(x_np: np.ndarray, step_dim: int = STEP_DIM):
    F = x_np.shape[1]
    S = int(np.ceil(F / step_dim))
    pad = S * step_dim - F
    if pad > 0:
        x_np = np.pad(x_np, ((0,0),(0,pad)), mode="constant")
    return x_np.reshape(-1, S, step_dim), S

# ===== 5) Mô hình LSTM + AttnPooling =====
class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.proj = nn.Linear(d, 1)
    def forward(self, H):        # H: [B,S,D]
        score = self.proj(H).squeeze(-1)           # [B,S]
        w = torch.softmax(score, dim=1)            # [B,S]
        pooled = (H * w.unsqueeze(-1)).sum(1)      # [B,D]
        return pooled

class LSTMTabular(nn.Module):
    def __init__(self, step_dim, hidden=256, layers=2, n_classes=10, dropout=0.2, bidir=True):
        super().__init__()
        self.step_dim = step_dim
        self.in_norm  = nn.LayerNorm(step_dim)
        self.lstm = nn.LSTM(
            input_size=step_dim,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout if layers > 1 else 0.0,
            bidirectional=bidir
        )
        d_out = hidden * (2 if bidir else 1)
        self.pool = AttnPool(d_out)
        self.head = nn.Sequential(
            nn.Linear(d_out, d_out//2), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(d_out//2, n_classes)
        )
    def forward(self, x):                       # x: [B, F]
        B, F = x.shape
        S = int(math.ceil(F / self.step_dim))
        pad = S * self.step_dim - F
        if pad > 0:
            x = torch.nn.functional.pad(x, (0, pad), value=0.0)
        x = x.view(B, S, self.step_dim)         # [B,S,step_dim]
        x = self.in_norm(x)
        H, _ = self.lstm(x)                     # [B,S,D]
        z = self.pool(H)                        # [B,D]
        return self.head(z)                     # [B,C]

model = LSTMTabular(step_dim=STEP_DIM, hidden=256, layers=2, n_classes=num_classes, dropout=0.15, bidir=True).to(device)

# ===== 6) Loss/optim/scheduler + mixup (công bằng) =====
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-4)

total_epochs = 120
warmup_epochs = 19
steps_per_epoch = max(1, len(train_loader))
def lr_lambda(step):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs * steps_per_epoch
    if step < warmup_steps: return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, (total_steps - warmup_steps))
    return 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def mixup_pair(x, y, alpha=0.10):
    if alpha <= 0: return x, y, 1.0, None
    lam  = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    x_m  = lam * x + (1 - lam) * x[perm]
    y_perm = y[perm]
    return x_m, y, lam, y_perm

# ===== 7) EMA (float-only) — device-safe =====
ema_decay = 0.995
ema_state = {}
for k, v in model.state_dict().items():
    if v.is_floating_point():
        ema_state[k] = v.detach().clone().to(device=v.device, dtype=v.dtype)

def ema_update(model, ema_state, decay: float):
    with torch.no_grad():
        for k, v in model.state_dict().items():
            if not v.is_floating_point(): continue
            if (ema_state[k].device != v.device) or (ema_state[k].dtype != v.dtype):
                ema_state[k] = ema_state[k].to(device=v.device, dtype=v.dtype)
            ema_state[k].mul_((decay)).add_(v.detach(), alpha=1.0 - decay)

def load_ema_state(model, ema_state):
    current = model.state_dict(); merged = {}
    for k, v in current.items():
        merged[k] = ema_state.get(k, v).to(device=v.device, dtype=v.dtype)
    model.load_state_dict(merged, strict=True)

# ===== 8) Train + Early-stopping theo Macro-F1 (VAL) =====
best_macro_f1, best_ema_snapshot, wait, patience = -1.0, None, 0, 20
global_step = 0

for epoch in range(total_epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        xb_m, ya, lam, yb_ = mixup_pair(xb, yb, alpha=0.10)
        logits = model(xb_m)
        loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb_) if yb_ is not None else criterion(logits, ya)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step(); scheduler.step()
        ema_update(model, ema_state, ema_decay)
        global_step += 1

    # Eval VAL với EMA
    saved_live = {k: v.detach().clone() for k, v in model.state_dict().items()}
    load_ema_state(model, ema_state)
    model.eval()
    with torch.no_grad():
        logits_val = []
        for xb, _ in val_loader:
            logits_val.append(model(xb.to(device)).cpu())
        logits_val = torch.cat(logits_val, 0).numpy()
    pred_val = logits_val.argmax(1)
    macro_f1 = f1_score(y_val, pred_val, average='macro', zero_division=0)

    if macro_f1 > best_macro_f1:
        best_macro_f1, wait = macro_f1, 0
        best_ema_snapshot = {k: v.detach().cpu().clone() for k, v in ema_state.items()}
    else:
        wait += 1

    model.load_state_dict(saved_live, strict=True)
    if wait >= patience:
        print(f"[Early-stop] epoch {epoch+1} | best VAL Macro-F1 = {best_macro_f1:.4f}")
        break

# Nạp EMA tốt nhất
if best_ema_snapshot is not None:
    ema_state = {k: t.clone().to(device) for k, t in best_ema_snapshot.items()}
    load_ema_state(model, ema_state)
model.eval()

# ===== 9) SUY LUẬN TEST =====
with torch.no_grad():
    logits_test = []
    for xb, _ in test_loader:
        logits_test.append(model(xb.to(device)).cpu())
    logits_test = torch.cat(logits_test, 0).numpy()

y_pred = logits_test.argmax(1)
# def softmax_np(z):
#     z = z - z.max(axis=1, keepdims=True)
#     e = np.exp(z)
#     return e / e.sum(axis=1, keepdims=True)

# # Xác suất dự đoán cho LSTM
# lstm_probs = softmax_np(logits_test)   # logits_test là numpy array từ bước suy luận LSTM
# lstm_preds = y_pred 

# ===== 10) Đánh giá =====
print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro',zero_division=0))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

Accuracy: 0.6918145956607495
Precision: 0.6849604325177503
DR: 0.6918145956607495
F1: 0.6679060522883344
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    130     39      0      0      0      0      0      0      0      0      0
  2      0     33    113      0      0      0      0     22      0      0      0      1
  3      0      0      0     11     39     44     75      0      0      0      0      0
  4      0      0      0      9     46     40     74      0      0      0      0      0
  5      0      0      0      3      6    157      3      0      0      0      0      0
  6      0      0      0      1     20     23    125      0      0      0      0      0
  7      0      0      1      0      0      0      0    129     39      0      0      0
  8      0      0      0      0      0      0      0     24    145      0      0      0
  9      0    

In [31]:
torch.save({
    "model_state": model.state_dict(),
    "ema_state": ema_state,
    "scaler": scaler,
    "config": {
        "step_dim": STEP_DIM,
        "num_classes": num_classes,
        "num_features": num_features,
    }
}, './models/aug_lstm.pkl')

# resDNN

In [32]:
import math, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, precision_recall_fscore_support

# ===== CỜ TUỲ CHỌN =====
ENABLE_HEAD_FINETUNE = True      # finetune riêng head vài epoch để nhạy với 7/10
ENABLE_OVR_BLEND     = False     # OvR cho 7/10 (LogReg) rồi blend vào xác suất DNN
OPT_BIAS_FOR_TARGETS = False     # True: tối ưu macro-F1 chỉ trên {7,10} khi chọn T & bias

# ===== Device & seed =====
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); np.random.seed(42)

# ===== Dữ liệu =====
X_full = train_df[FS].values
y_full = train_df['Label'].values
X_test = test_df[FS].values
y_test = test_df["Label"].values

num_classes  = int(max(y_full.max(), y_test.max())) + 1
num_features = X_full.shape[1]

# ===== 1) Stratified split: train_in / val (15%) =====
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss.split(X_full, y_full))
X_tr_in, y_tr_in = X_full[train_idx], y_full[train_idx]
X_val,   y_val   = X_full[val_idx],   y_full[val_idx]

# ===== 2) Scaler =====
scaler = StandardScaler()
X_tr_in_sc = scaler.fit_transform(X_tr_in)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# ===== 3) Sampler cân bằng + boost 7/10 =====
counts = np.bincount(y_tr_in, minlength=num_classes).astype(np.float32)
inv = counts.sum() / (counts + 1e-9)
inv /= inv.mean()
boost = np.ones(num_classes, dtype=np.float32)
boost[[7,10]] = 1.6  # tăng tần suất gặp 7/10 (giữ hệ số)
sample_w = (inv * boost)[y_tr_in]
sampler  = WeightedRandomSampler(weights=sample_w, num_samples=len(sample_w), replacement=True)

# ===== 4) DataLoaders =====
Xtr_t = torch.tensor(X_tr_in_sc, dtype=torch.float32)
ytr_t = torch.tensor(y_tr_in,    dtype=torch.long)
Xva_t = torch.tensor(X_val_sc,   dtype=torch.float32)
yva_t = torch.tensor(y_val,      dtype=torch.long)
Xte_t = torch.tensor(X_test_sc,  dtype=torch.float32)
yte_t = torch.tensor(y_test,     dtype=torch.long)

batch_size = 1024
train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, sampler=sampler, pin_memory=True)
val_loader   = DataLoader(TensorDataset(Xva_t, yva_t), batch_size=4096, shuffle=False, pin_memory=True)
test_loader  = DataLoader(TensorDataset(Xte_t, yte_t), batch_size=4096, shuffle=False, pin_memory=True)

# ===== 5) Mô hình =====
class ResidualBlock(nn.Module):
    def __init__(self, d_in, d_hid, p=0.25):
        super().__init__()
        self.lin1 = nn.Linear(d_in, d_hid)
        self.bn1  = nn.BatchNorm1d(d_hid)
        self.lin2 = nn.Linear(d_hid, d_in)
        self.ln2  = nn.LayerNorm(d_in)
        self.drop = nn.Dropout(p)
    def forward(self, x):
        h = self.drop(torch.relu(self.bn1(self.lin1(x))))
        h = self.lin2(h)
        return torch.relu(self.ln2(x + h))

class ResDNN(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        W = 512
        self.stem = nn.Sequential(nn.Linear(in_dim, W), nn.BatchNorm1d(W), nn.ReLU(), nn.Dropout(0.30))
        self.block1 = ResidualBlock(W, W//2, p=0.30)
        self.block2 = ResidualBlock(W, W//2, p=0.25)
        self.block3 = ResidualBlock(W, W//2, p=0.20)
        self.head   = nn.Linear(W, n_classes)
    def forward(self, x):
        h = self.stem(x)
        h = self.block1(h); h = self.block2(h); h = self.block3(h)
        return self.head(h)

model = ResDNN(num_features, num_classes).to(device)

# ===== 6) CB-Focal Loss + class weights (Effective Number) =====
def cb_class_weights(counts, beta=0.999):
    eff = (1.0 - beta) / (1.0 - np.power(beta, counts + 1e-12))
    w   = eff / eff.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)

class FocalCE(nn.Module):
    def __init__(self, gamma=1.7, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight
    def forward(self, logits, target):
        logp = torch.log_softmax(logits, dim=1)
        p    = torch.exp(logp)
        idx  = torch.arange(logits.size(0), device=logits.device)
        pt   = p[idx, target]
        loss = -((1-pt) ** self.gamma) * logp[idx, target]
        if self.weight is not None:
            loss = loss * self.weight[target]
        return loss.mean()

cls_w  = cb_class_weights(counts, beta=0.999)
criterion = FocalCE(gamma=1.7, weight=cls_w)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-4)

# ===== 7) Lịch LR: warmup + cosine =====
total_epochs  = 120
warmup_epochs = 8
steps_per_epoch = max(1, len(train_loader))
def lr_lambda(current_step):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs * steps_per_epoch
    if current_step < warmup_steps:
        return (current_step + 1) / warmup_steps
    progress = (current_step - warmup_steps) / max(1, (total_steps - warmup_steps))
    return 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ===== 8) Mixup (giảm khi batch có 7/10) =====
def mixup_dynamic(x, y, base_alpha=0.10):
    alpha = base_alpha
    if any(lbl in (7,10) for lbl in y.tolist()):
        alpha = 0.02
    if alpha <= 0: return x, y, 1.0, None
    lam  = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    x_m  = lam * x + (1 - lam) * x[perm]
    y_perm = y[perm]
    return x_m, y, lam, y_perm

# ===== 9) EMA (float-only) — DEVICE SAFE =====
ema_decay = 0.995
ema_state = {}
for k, v in model.state_dict().items():
    if v.is_floating_point():
        ema_state[k] = v.detach().clone().to(device=v.device, dtype=v.dtype)

def ema_update(model, ema_state, decay: float):
    with torch.no_grad():
        for k, v in model.state_dict().items():
            if not v.is_floating_point():
                continue
            if (ema_state[k].device != v.device) or (ema_state[k].dtype != v.dtype):
                ema_state[k] = ema_state[k].to(device=v.device, dtype=v.dtype)
            ema_state[k].mul_(decay).add_(v.detach(), alpha=1.0 - decay)

def load_ema_state(model, ema_state):
    current = model.state_dict()
    merged = {}
    for k, v in current.items():
        if k in ema_state:
            merged[k] = ema_state[k].to(device=v.device, dtype=v.dtype)
        else:
            merged[k] = v
    model.load_state_dict(merged, strict=True)

# ===== 10) Train + Early-stopping Macro-F1 (VAL) =====
best_f1, best_ema_snapshot, wait, patience = -1.0, None, 0, 20
global_step = 0

for epoch in range(total_epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        xb_m, ya, lam, yb_ = mixup_dynamic(xb, yb, base_alpha=0.10)
        logits = model(xb_m)
        loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb_) if yb_ is not None else criterion(logits, ya)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()
        scheduler.step()
        ema_update(model, ema_state, ema_decay)
        global_step += 1

    # --- Eval VAL với EMA ---
    saved_live = {k: v.detach().clone() for k, v in model.state_dict().items()}
    load_ema_state(model, ema_state)
    model.eval()
    with torch.no_grad():
        logits_val = []
        for xb, _ in val_loader:
            logits_val.append(model(xb.to(device)).cpu())
        logits_val = torch.cat(logits_val, 0).numpy()
    pred_val = logits_val.argmax(1)
    f1_val_all = f1_score(y_val, pred_val, average='macro')
    f1_val_target = f1_score(y_val, pred_val, labels=[7,10], average='macro', zero_division=0)
    f1_val = f1_val_target if OPT_BIAS_FOR_TARGETS else f1_val_all

    if f1_val > best_f1:
        best_f1, wait = f1_val, 0
        best_ema_snapshot = {k: v.detach().cpu().clone() for k, v in ema_state.items()}
    else:
        wait += 1

    model.load_state_dict(saved_live, strict=True)

    if wait >= patience:
        print(f"[Early-stop] epoch {epoch+1} | best VAL Macro-F1 ({'targets' if OPT_BIAS_FOR_TARGETS else 'all'}) = {best_f1:.4f}")
        break

# ===== 11) Load EMA tốt nhất và (tuỳ chọn) finetune head =====
if best_ema_snapshot is not None:
    ema_state = {k: t.clone().to(device) for k, t in best_ema_snapshot.items()}
    load_ema_state(model, ema_state)

if ENABLE_HEAD_FINETUNE:
    for p in model.stem.parameters():   p.requires_grad = False
    for p in model.block1.parameters(): p.requires_grad = False
    for p in model.block2.parameters(): p.requires_grad = False
    for p in model.block3.parameters(): p.requires_grad = False
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=5e-4, weight_decay=1e-4)
    ft_epochs, ft_patience, ft_wait, best_ft = 12, 6, 0, -1.0
    for epoch in range(ft_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)  # tắt mixup khi finetune head
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.head.parameters(), 3.0)
            optimizer.step()
            ema_update(model, ema_state, ema_decay)
        # eval VAL
        saved_live = {k: v.detach().clone() for k, v in model.state_dict().items()}
        load_ema_state(model, ema_state)
        model.eval()
        with torch.no_grad():
            logits_val = []
            for xb, _ in val_loader:
                logits_val.append(model(xb.to(device)).cpu())
            logits_val = torch.cat(logits_val, 0).numpy()
        pred_val = logits_val.argmax(1)
        f1_val_all = f1_score(y_val, pred_val, average='macro')
        f1_val_target = f1_score(y_val, pred_val, labels=[7,10], average='macro', zero_division=0)
        f1_val = f1_val_target if OPT_BIAS_FOR_TARGETS else f1_val_all
        if f1_val > best_ft:
            best_ft, ft_wait = f1_val, 0
            best_ema_snapshot = {k: v.detach().cpu().clone() for k, v in ema_state.items()}
        else:
            ft_wait += 1
        model.load_state_dict(saved_live, strict=True)
        if ft_wait >= ft_patience:
            print(f"[Finetune head Early-stop] epoch {epoch+1} | best VAL Macro-F1 = {best_ft:.4f}")
            break
    if best_ema_snapshot is not None:
        ema_state = {k: t.clone().to(device) for k, t in best_ema_snapshot.items()}
        load_ema_state(model, ema_state)

model.eval()

# ===== 12) Temperature scaling (VAL) =====
def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

with torch.no_grad():
    logits_val = []
    for xb, _ in val_loader:
        logits_val.append(model(xb.to(device)).cpu())
    logits_val = torch.cat(logits_val, 0).numpy()
    logits_test = []
    for xb, _ in test_loader:
        logits_test.append(model(xb.to(device)).cpu())
    logits_test = torch.cat(logits_test, 0).numpy()

best_T, best_val_f1 = 1.0, -1.0
for T in np.arange(0.5, 3.05, 0.1):
    p_val = softmax_np(logits_val / T)
    pred  = p_val.argmax(1)
    f1_all    = f1_score(y_val, pred, average='macro')
    f1_target = f1_score(y_val, pred, labels=[7,10], average='macro', zero_division=0)
    f1_use    = f1_target if OPT_BIAS_FOR_TARGETS else f1_all
    if f1_use > best_val_f1:
        best_val_f1, best_T = f1_use, float(T)

# ===== 13) (Tuỳ chọn) OvR blend cho 7/10 =====
if ENABLE_OVR_BLEND:
    from sklearn.linear_model import LogisticRegression
    X_ovr = X_tr_in_sc; y_ovr = y_tr_in
    ovr_targets = [7,10]
    lr_models = {}
    for t in ovr_targets:
        y_bin = (y_ovr == t).astype(int)
        lr = LogisticRegression(max_iter=200)
        lr.fit(X_ovr, y_bin)
        lr_models[t] = lr
    ovr_scores_test = {t: lr_models[t].predict_proba(X_test_sc)[:,1] for t in ovr_targets}

# ===== 14) Logit-bias search trên VAL để đẩy 7/10 =====
targets = [7,10]
grid = np.arange(0.0, 1.30, 0.10)
best_b, best_f1_bias = np.zeros(num_classes), -1.0

for b7 in grid:
    for b10 in grid:
        b = np.zeros(num_classes, dtype=np.float32)
        b[7], b[10] = b7, b10
        p_val = softmax_np((logits_val / best_T) + b[None, :])
        pred = p_val.argmax(1)
        f1_all    = f1_score(y_val, pred, average='macro')
        f1_target = f1_score(y_val, pred, labels=targets, average='macro', zero_division=0)
        f1_use    = f1_target if OPT_BIAS_FOR_TARGETS else f1_all
        if f1_use > best_f1_bias:
            best_f1_bias, best_b = f1_use, b.copy()

# ===== 15) Suy luận TEST với T* và bias* (và OvR nếu bật) =====
p_test = softmax_np((logits_test / best_T) + best_b[None, :])

if ENABLE_OVR_BLEND:
    for t in [7,10]:
        s = ovr_scores_test[t]  # [N]
        p_test[:, t] *= (0.5 + 0.5 * s)
    p_test = p_test / p_test.sum(axis=1, keepdims=True)

y_pred = p_test.argmax(1)
# dnn_probs = p_test                                   # shape: (n_samples, n_classes)
# dnn_preds = y_pred
# ===== 16) Đánh giá =====
print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
# labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

[Finetune head Early-stop] epoch 7 | best VAL Macro-F1 = 0.7309
Accuracy: 0.7041420118343196
Precision: 0.7176527184580589
DR: 0.7041420118343197
F1: 0.6915452403945634
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    139     30      0      0      0      0      0      0      0      0      0
  2      0     19    150      0      0      0      0      0      0      0      0      0
  3      0      0      0     58     17     24     70      0      0      0      0      0
  4      0      0      0     28     58     11     72      0      0      0      0      0
  5      0      0      0     20     12    134      3      0      0      0      0      0
  6      0      0      0      4     17     10    138      0      0      0      0      0
  7      0      1     76      0      0      0      0     65     27      0      0      0
  8      0      0      0      0      0

In [34]:
torch.save({
    "model_state": model.state_dict(),   # trọng số cuối (đã nạp EMA tốt nhất)
    "ema_state": ema_state,              # lưu lại EMA state để khôi phục
    "scaler": scaler,                    # StandardScaler để inference sau này
    "config": {
        "num_classes": num_classes,
        "num_features": num_features,
        "enable_head_finetune": ENABLE_HEAD_FINETUNE,
        "opt_bias_for_targets": OPT_BIAS_FOR_TARGETS,
        "enable_ovr_blend": ENABLE_OVR_BLEND,
    }
}, './models/aug_dnn.pkl')

# Inference

## load models

In [22]:
X_test = test_df[FS]
y_test = test_df['Label']

xgb = joblib.load('./models/aug_xgb.pkl')
cat = joblib.load('./models/aug_cat.pkl')
rf = joblib.load('./models/aug_rf.pkl')
et = joblib.load('./models/aug_et.pkl')
lgbm = joblib.load('./models/aug_lgbm.pkl')

In [35]:
# DNN model
ckpt = torch.load("./models/aug_dnn.pkl", map_location=device)
dnn = ResDNN(
    in_dim=ckpt["config"]["num_features"],
    n_classes=ckpt["config"]["num_classes"]
).to(device)

dnn.load_state_dict(ckpt["model_state"])
dnn.eval()
scaler = ckpt["scaler"]

X_test_scaled = scaler.transform(X_test)
# Chuyển sang tensor
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)
with torch.no_grad():
    logits = dnn(X_test_tensor)
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)
dnn_probs = probs.cpu().numpy()
dnn_preds = preds.cpu().numpy()


# biLSTM
ckpt = torch.load("./models/aug_lstm.pkl", map_location=device)
lstm = LSTMTabular(
    step_dim=ckpt["config"]["step_dim"],
    n_classes=ckpt["config"]["num_classes"]
).to(device)

lstm.load_state_dict(ckpt["model_state"])
lstm.eval()
scaler = ckpt["scaler"]
X_test_scaled = scaler.transform(X_test)
# Chuyển sang tensor
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)
with torch.no_grad():
    logits = lstm(X_test_tensor)
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)
lstm_probs = probs.cpu().numpy()
lstm_preds = preds.cpu().numpy()

/tmp/ipykernel_151758/1397492213.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load("./models/aug_dnn.pkl", map_location=device)
/tmp/ipykernel_151758/139

In [36]:
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=lstm_preds, average='macro'))

DR: 0.6918145956607495


In [24]:
# --- Predict proba ---
xgb_probs = xgb.predict_proba(X_test)
cat_probs = cat.predict_proba(X_test)
rf_probs  = rf.predict_proba(X_test)
et_probs  = et.predict_proba(X_test)
lgbm_probs = lgbm.predict_proba(X_test)

# --- Predict labels ---
xgb_preds = xgb.predict(X_test)
cat_preds = cat.predict(X_test)
rf_preds  = rf.predict(X_test)
et_preds  = et.predict(X_test)
lgbm_preds = lgbm.predict(X_test)

## weighted 

In [73]:
import numpy as np
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import label_binarize

# ====== CHỌN MỤC TIÊU ======
# 'logloss' -> khuyến nghị; 'auc' -> macro OvR
objective = 'auc'   # hoặc 'auc'

# ====== INPUTS ======
# Có sẵn 7 dự đoán xác suất: 
preds_list = [xgb_probs, cat_probs, lgbm_probs, et_probs, rf_probs, lstm_probs, dnn_probs] 
y_val = y_test  # ground true

# ====== CHUẨN HOÁ SHAPE & HÀNG TỔNG = 1 ======
_fixed = []
for p in preds_list:
    p = np.asarray(p)
    if p.ndim == 1:
        # binary: vector prob class1 -> (n,2)
        p = np.column_stack([1.0 - p, p])
    elif p.ndim == 2 and p.shape[1] == 1:
        # binary: (n,1) -> (n,2)
        p = np.column_stack([1.0 - p[:, 0], p[:, 0]])
    # normalize mỗi hàng
    row_sum = p.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    p = p / row_sum
    _fixed.append(p)
preds_list = _fixed

n_classes = preds_list[0].shape[1]
for idx, p in enumerate(preds_list):
    assert p.shape[1] == n_classes, f"Model {idx} có C={p.shape[1]} khác {n_classes}"

# One-hot cho AUC macro nếu multiclass
if n_classes > 2:
    Y_bin = label_binarize(y_val, classes=np.arange(n_classes))

# ====== HÀM PHỤ TRỢ (projection lên simplex) ======
def project_simplex(v):
    """
    Euclidean projection of v onto the probability simplex {w | w>=0, sum(w)=1}
    """
    v = np.asarray(v, dtype=float)
    if v.sum() == 1.0 and np.all(v >= 0):
        return v
    # Thuật toán: sort giảm, tìm theta
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u)
    rho = np.nonzero(u * np.arange(1, len(u)+1) > (cssv - 1))[0][-1]
    theta = (cssv[rho] - 1.0) / (rho + 1)
    w = np.maximum(v - theta, 0.0)
    return w

# ====== HÀM ĐÁNH GIÁ THEO MỤC TIÊU ======
def evaluate_objective(weights):
    # kết hợp xác suất
    proba = np.zeros_like(preds_list[0])
    for w, p in zip(weights, preds_list):
        proba += w * p
    if objective == 'logloss':
        return log_loss(y_val, proba), proba  # minimize
    else:  # 'auc'
        if n_classes == 2:
            auc = roc_auc_score(y_val, proba[:, 1])
        else:
            auc = roc_auc_score(Y_bin, proba, average='macro', multi_class='ovr')
        return -auc, proba  # minimize -AUC

# ====== 1) TỐI ƯU LIÊN TỤC: LOGLOSS BẰNG PROJECTED GRADIENT DESCENT ======
best_weights = None
best_obj = np.inf
best_proba = None

if objective == 'logloss':
    # PGD cấu hình
    rng = np.random.default_rng(42)
    restarts = 8            # số khởi tạo ngẫu nhiên
    max_iter = 800          # số vòng lặp PGD
    lr = 0.5                # learning rate ban đầu
    lr_decay = 0.995        # giảm LR mỗi vòng
    eps = 1e-12             # tránh chia 0

    # Khởi tạo thêm từ Dirichlet (đều, thiên về mô hình mạnh)
    inits = [np.ones(7)/7,
             rng.dirichlet(alpha=np.ones(7)*1.0),
             rng.dirichlet(alpha=np.array([3,2,2,3,2,1,2], dtype=float)),
             rng.dirichlet(alpha=np.array([4,1,1,4,1,1,3], dtype=float))]
    while len(inits) < restarts:
        inits.append(rng.dirichlet(alpha=np.ones(7)))

    for w0 in inits:
        w = w0.copy()
        cur_lr = lr
        # PGD loop
        for t in range(max_iter):
            # ensemble xác suất
            proba = np.zeros_like(preds_list[0])
            for wi, pi in zip(w, preds_list):
                proba += wi * pi
            # lấy proba của lớp đúng cho từng mẫu
            # idx: vector chỉ số lớp đúng
            idx = np.asarray(y_val, dtype=int)
            p_y = proba[np.arange(proba.shape[0]), idx]  # (n,)
            # gradient với mỗi weight i: d/dw_i [-1/n sum log p_y] = -1/n sum (p_i_y / p_y)
            g = np.zeros_like(w)
            for i in range(len(w)):
                p_i_y = preds_list[i][np.arange(proba.shape[0]), idx]
                g[i] = - np.mean(p_i_y / np.clip(p_y, eps, None))
            # bước cập nhật
            w = w - cur_lr * g
            # project lên simplex
            w = project_simplex(w)
            # giảm lr
            cur_lr *= lr_decay

            # early check mỗi 50 bước
            if (t+1) % 50 == 0:
                obj, proba_eval = evaluate_objective(w)
                if obj < best_obj:
                    best_obj = obj
                    best_weights = w.copy()
                    best_proba = proba_eval.copy()

    print("[LOGLOSS] Best weights:", best_weights, "sum =", best_weights.sum())
    print("[LOGLOSS] Best LogLoss:", best_obj)

# ====== 2) TỐI ƯU LIÊN TỤC: AUC BẰNG DIRICHLET RANDOM SEARCH + COORDINATE ASCENT ======
if objective == 'auc':
    rng = np.random.default_rng(123)
    samples = 4000              # số mẫu Dirichlet
    alphas = [
        np.ones(7)*1.0,         # đều
        np.array([3,2,2,3,2,1,2], dtype=float),  # ưu tiên xgb/lgbm/dnn chút
        np.array([4,1,1,4,1,1,3], dtype=float)
    ]
    best_obj = np.inf
    best_weights = None
    best_proba = None

    # Dirichlet sampling
    for alpha in alphas:
        for _ in range(samples // len(alphas)):
            w = rng.dirichlet(alpha)
            obj, proba = evaluate_objective(w)  # obj = -AUC
            if obj < best_obj:
                best_obj = obj
                best_weights = w.copy()
                best_proba = proba.copy()

    # Coordinate ascent tinh chỉnh quanh nghiệm tốt nhất
    delta = 0.10
    min_delta = 0.005
    patience = 0
    max_patience = 10
    while delta >= min_delta and patience <= max_patience:
        improved = False
        for i in range(7):
            w = best_weights.copy()
            # thử tăng weight i
            inc = min(delta, 1.0 - w[i])
            if inc > 0:
                w[i] += inc
                # giảm đều từ các trọng số khác (giữ sum=1, w>=0)
                mask = np.ones(7, dtype=bool); mask[i] = False
                dec_pool = w[mask].sum()
                if dec_pool > 0:
                    w[mask] = np.maximum(w[mask] * (1 - inc/dec_pool), 0.0)
                w = project_simplex(w)
                obj_inc, proba_inc = evaluate_objective(w)
            else:
                obj_inc = np.inf

            # thử giảm weight i
            w2 = best_weights.copy()
            dec = min(delta, w2[i])
            if dec > 0:
                w2[i] -= dec
                mask2 = np.ones(7, dtype=bool); mask2[i] = False
                w2[mask2] = project_simplex(w2[mask2] / w2[mask2].sum()) * (1 - w2[i])
                obj_dec, proba_dec = evaluate_objective(w2)
            else:
                obj_dec = np.inf

            # chọn cải thiện tốt hơn
            if obj_inc < best_obj or obj_dec < best_obj:
                if obj_inc <= obj_dec:
                    best_obj = obj_inc
                    best_weights = w
                    best_proba = proba_inc
                else:
                    best_obj = obj_dec
                    best_weights = w2
                    best_proba = proba_dec
                improved = True

        if improved:
            patience = 0
        else:
            patience += 1
            delta *= 0.7  # giảm bước tinh chỉnh

    print("[AUC] Best weights:", best_weights, "sum =", best_weights.sum())
    print("[AUC] Best AUC macro OvR:", -best_obj)  # vì obj = -AUC

# ====== ĐÁNH GIÁ CHI TIẾT TẠI TRỌNG SỐ TỐT NHẤT ======
y_pred = best_proba.argmax(axis=1)

print("Accuracy:", accuracy_score(y_true=y_val, y_pred=y_pred))
print("Precision (macro):", precision_score(y_true=y_val, y_pred=y_pred, average='macro', zero_division=0))
print("Recall (macro):",   recall_score(y_true=y_val, y_pred=y_pred, average='macro', zero_division=0))
print("F1 (macro):",       f1_score(y_true=y_val, y_pred=y_pred, average='macro', zero_division=0))

cm = confusion_matrix(y_val, y_pred, labels=labels)
print("\nConfusion matrix (rows=true, cols=pred):")
print("     " + " ".join(f"{int(lbl):6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{int(lbl):3d} " + " ".join(f"{int(val):6d}" for val in row))


[AUC] Best weights: [0.1787166  0.         0.0147797  0.01030202 0.1734231  0.43584224
 0.18693633] sum = 0.9999999999999999
[AUC] Best AUC macro OvR: 0.9927804709324963
Accuracy: 0.903353057199211
Precision (macro): 0.9016828263508615
Recall (macro): 0.9033530571992111
F1 (macro): 0.9023008792106607

Confusion matrix (rows=true, cols=pred):
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      2    165      0      0      0      0      2      0      0      0      0
  3      0      0      0    167      0      2      0      0      0      0      0      0
  4      0      0      0      0    168      1      0      0      0      0      0      0
  5      0      0      0      0      1    168      0      0      0      0      0      0
  6      0      0      0      0      0 

In [48]:
import numpy as np
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import label_binarize

# ====== CHỌN MỤC TIÊU ======
# 'logloss' -> khuyến nghị; 'auc' -> macro OvR
objective = 'auc'   # hoặc 'logloss'

# ====== INPUTS ======
# Chỉ dùng 5 dự đoán: XGB, ExtraTree, RF, DNN, BiLSTM (thứ tự quan trọng để tùy chỉnh alpha nếu muốn)
preds_list = [xgb_probs, et_probs, rf_probs, cat_probs, lstm_probs]
# Nhãn thật (n_samples,), dùng tập validation để tối ưu
y_val = y_test  # thay bằng biến của bạn

# ====== CHUẨN HOÁ SHAPE & HÀNG TỔNG = 1 ======
_fixed = []
for p in preds_list:
    p = np.asarray(p)
    if p.ndim == 1:
        # binary: vector prob class1 -> (n,2)
        p = np.column_stack([1.0 - p, p])
    elif p.ndim == 2 and p.shape[1] == 1:
        # binary: (n,1) -> (n,2)
        p = np.column_stack([1.0 - p[:, 0], p[:, 0]])
    # normalize mỗi hàng
    row_sum = p.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    p = p / row_sum
    _fixed.append(p)
preds_list = _fixed

M = len(preds_list)      # số mô hình thực tế (here = 5)
n_classes = preds_list[0].shape[1]
for idx, p in enumerate(preds_list):
    assert p.shape[1] == n_classes, f"Model {idx} có C={p.shape[1]} khác {n_classes}"

# One-hot cho AUC macro nếu multiclass
if n_classes > 2:
    Y_bin = label_binarize(y_val, classes=np.arange(n_classes))

# Nếu 'labels' chưa có, đặt mặc định
try:
    labels
except NameError:
    labels = np.arange(n_classes)

# ====== HÀM PHỤ TRỢ (projection lên simplex) ======
def project_simplex(v):
    """
    Euclidean projection of v onto the probability simplex {w | w>=0, sum(w)=1}
    """
    v = np.asarray(v, dtype=float)
    if np.all(v >= 0) and abs(v.sum() - 1.0) < 1e-12:
        return v
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u)
    rho_arr = u * np.arange(1, len(u) + 1) > (cssv - 1)
    if not np.any(rho_arr):
        # fallback: project to uniform
        return np.ones_like(v) / len(v)
    rho = np.nonzero(rho_arr)[0][-1]
    theta = (cssv[rho] - 1.0) / (rho + 1)
    w = np.maximum(v - theta, 0.0)
    return w

# ====== HÀM ĐÁNH GIÁ THEO MỤC TIÊU ======
def evaluate_objective(weights):
    # kết hợp xác suất
    proba = np.zeros_like(preds_list[0])
    for w, p in zip(weights, preds_list):
        proba += w * p
    if objective == 'logloss':
        return log_loss(y_val, proba), proba  # minimize
    else:  # 'auc'
        if n_classes == 2:
            auc = roc_auc_score(y_val, proba[:, 1])
        else:
            auc = roc_auc_score(Y_bin, proba, average='macro', multi_class='ovr')
        return -auc, proba  # minimize -AUC

# ====== 1) TỐI ƯU LIÊN TỤC: LOGLOSS BẰNG PROJECTED GRADIENT DESCENT ======
best_weights = None
best_obj = np.inf
best_proba = None

if objective == 'logloss':
    # PGD cấu hình
    rng = np.random.default_rng(42)
    restarts = 8            # số khởi tạo ngẫu nhiên
    max_iter = 800          # số vòng lặp PGD
    lr = 0.5                # learning rate ban đầu
    lr_decay = 0.995        # giảm LR mỗi vòng
    eps = 1e-12             # tránh chia 0

    # Khởi tạo thêm từ Dirichlet (đều, thiên về mô hình mạnh)
    # Tùy chỉnh alpha vector có độ dài M
    inits = [np.ones(M)/M,
             rng.dirichlet(alpha=np.ones(M)*1.0),
             rng.dirichlet(alpha=np.array([3 if i==0 else 2 for i in range(M)], dtype=float)),
             rng.dirichlet(alpha=np.array([4 if i==0 else 1 for i in range(M)], dtype=float))]
    while len(inits) < restarts:
        inits.append(rng.dirichlet(alpha=np.ones(M)))

    for w0 in inits:
        w = w0.copy()
        cur_lr = lr
        # PGD loop
        for t in range(max_iter):
            # ensemble xác suất
            proba = np.zeros_like(preds_list[0])
            for wi, pi in zip(w, preds_list):
                proba += wi * pi
            # lấy proba của lớp đúng cho từng mẫu
            idx = np.asarray(y_val, dtype=int)
            p_y = proba[np.arange(proba.shape[0]), idx]  # (n,)
            # gradient với mỗi weight i: d/dw_i [-1/n sum log p_y] = -1/n sum (p_i_y / p_y)
            g = np.zeros_like(w)
            for i in range(len(w)):
                p_i_y = preds_list[i][np.arange(proba.shape[0]), idx]
                g[i] = - np.mean(p_i_y / np.clip(p_y, eps, None))
            # bước cập nhật
            w = w - cur_lr * g
            # project lên simplex
            w = project_simplex(w)
            # giảm lr
            cur_lr *= lr_decay

            # early check mỗi 50 bước
            if (t+1) % 50 == 0:
                obj, proba_eval = evaluate_objective(w)
                if obj < best_obj:
                    best_obj = obj
                    best_weights = w.copy()
                    best_proba = proba_eval.copy()

    print("[LOGLOSS] Best weights:", best_weights, "sum =", best_weights.sum())
    print("[LOGLOSS] Best LogLoss:", best_obj)

# ====== 2) TỐI ƯU LIÊN TỤC: AUC BẰNG DIRICHLET RANDOM SEARCH + COORDINATE ASCENT ======
if objective == 'auc':
    rng = np.random.default_rng(123)
    samples = 4000              # số mẫu Dirichlet
    # Danh sách alphas (mỗi mục có chiều M)
    alphas = [
        np.ones(M)*1.0,         # đều
        # ưu tiên XGB (index 0) và RF (index 2) & DNN (index 3) ví dụ
        np.array([3 if i==0 else (2 if i in (2,3) else 1) for i in range(M)], dtype=float),
        np.array([4 if i==0 else (3 if i==2 else 1) for i in range(M)], dtype=float)
    ]
    best_obj = np.inf
    best_weights = None
    best_proba = None

    # Dirichlet sampling
    for alpha in alphas:
        for _ in range(samples // len(alphas)):
            w = rng.dirichlet(alpha)
            obj, proba = evaluate_objective(w)  # obj = -AUC
            if obj < best_obj:
                best_obj = obj
                best_weights = w.copy()
                best_proba = proba.copy()

    # Coordinate ascent tinh chỉnh quanh nghiệm tốt nhất
    delta = 0.10
    min_delta = 0.005
    patience = 0
    max_patience = 10
    while delta >= min_delta and patience <= max_patience:
        improved = False
        for i in range(M):
            w = best_weights.copy()
            # thử tăng weight i
            inc = min(delta, 1.0 - w[i])
            if inc > 0:
                w[i] += inc
                # giảm đều từ các trọng số khác (giữ sum=1, w>=0)
                mask = np.ones(M, dtype=bool); mask[i] = False
                dec_pool = w[mask].sum()
                if dec_pool > 0:
                    w[mask] = np.maximum(w[mask] * (1 - inc/dec_pool), 0.0)
                w = project_simplex(w)
                obj_inc, proba_inc = evaluate_objective(w)
            else:
                obj_inc = np.inf
                proba_inc = None

            # thử giảm weight i
            w2 = best_weights.copy()
            dec = min(delta, w2[i])
            if dec > 0:
                w2[i] -= dec
                mask2 = np.ones(M, dtype=bool); mask2[i] = False
                # renormalize the remaining to sum to (1 - w2[i])
                rem = w2[mask2]
                rem_sum = rem.sum()
                if rem_sum > 0:
                    rem = rem / rem_sum * (1.0 - w2[i])
                w2[mask2] = rem
                w2 = project_simplex(w2)
                obj_dec, proba_dec = evaluate_objective(w2)
            else:
                obj_dec = np.inf
                proba_dec = None

            # chọn cải thiện tốt hơn
            if obj_inc < best_obj or obj_dec < best_obj:
                if obj_inc <= obj_dec:
                    best_obj = obj_inc
                    best_weights = w
                    best_proba = proba_inc
                else:
                    best_obj = obj_dec
                    best_weights = w2
                    best_proba = proba_dec
                improved = True

        if improved:
            patience = 0
        else:
            patience += 1
            delta *= 0.7  # giảm bước tinh chỉnh

    print("[AUC] Best weights:", best_weights, "sum =", best_weights.sum())
    print("[AUC] Best AUC macro OvR:", -best_obj)  # vì obj = -AUC

# ====== ĐÁNH GIÁ CHI TIẾT TẠI TRỌNG SỐ TỐT NHẤT ======
proba_best = best_proba
y_pred = proba_best.argmax(axis=1)

print("Accuracy:", accuracy_score(y_true=y_val, y_pred=y_pred))
print("Precision (macro):", precision_score(y_true=y_val, y_pred=y_pred, average='macro', zero_division=0))
print("Recall (macro):",   recall_score(y_true=y_val, y_pred=y_pred, average='macro', zero_division=0))
print("F1 (macro):",       f1_score(y_true=y_val, y_pred=y_pred, average='macro', zero_division=0))

cm = confusion_matrix(y_val, y_pred, labels=labels)
print("\nConfusion matrix (rows=true, cols=pred):")
print("     " + " ".join(f"{int(lbl):6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{int(lbl):3d} " + " ".join(f"{int(val):6d}" for val in row))


[AUC] Best weights: [0.20051262 0.07406998 0.16583603 0.         0.55958136] sum = 0.9999999999999999
[AUC] Best AUC macro OvR: 0.992685246781742
Accuracy: 0.895956607495069
Precision (macro): 0.8978536351803816
Recall (macro): 0.8959566074950692
F1 (macro): 0.8962446941903194

Confusion matrix (rows=true, cols=pred):
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      2    151      0      0      0      0     16      0      0      0      0
  3      0      0      0    167      0      2      0      0      0      0      0      0
  4      0      0      0      0    168      1      0      0      0      0      0      0
  5      0      0      0      0      1    168      0      0      0      0      0      0
  6      0      0      0      0      0      1    168      0    

In [65]:
import numpy as np
from sklearn.metrics import (
    f1_score, recall_score, accuracy_score, precision_score, confusion_matrix
)

# ====== cấu hình ======
metric = "f1"      # "f1" hoặc "recall"
step = 0.01
N = int(round(1/step))

# ====== proba của 2 mô hình ======
preds_list = [et_probs, dnn_probs] # , dnn_probs

best_score = -1.0
best_weights = None

# ====== search weights cho 2 mô hình ======
for i in range(N + 1):
    w1 = i / N
    w2 = 1.0 - w1

    proba = w1 * preds_list[0] + w2 * preds_list[1]
    y_pred = proba.argmax(axis=1)

    if metric == "f1":
        score = f1_score(y_test, y_pred, average='macro', zero_division=0)
    else:
        score = recall_score(y_test, y_pred, average='macro', zero_division=0)

    if score > best_score:
        best_score = score
        best_weights = (w1, w2)

print("Best weights (w1, w2):", best_weights, "sum =", sum(best_weights))
print(f"Best {metric} (macro):", best_score)

# ====== đánh giá chi tiết tại trọng số tốt nhất ======
w1, w2 = best_weights
proba_best =proba = w1 * preds_list[0] + w2 * preds_list[1]
y_pred = proba_best.argmax(axis=1)

print("Accuracy:", accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision (macro):", precision_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("Recall (macro):", recall_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("F1 (macro):", f1_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
cm = confusion_matrix(y_val, y_pred, labels=labels)
print("\nConfusion matrix (rows=true, cols=pred):")
print("     " + " ".join(f"{int(lbl):6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{int(lbl):3d} " + " ".join(f"{int(val):6d}" for val in row))

Best weights (w1, w2): (0.29, 0.71) sum = 1.0
Best f1 (macro): 0.8617737132021092
Accuracy: 0.8653846153846154
Precision (macro): 0.8643982010549444
Recall (macro): 0.8653846153846154
F1 (macro): 0.8617737132021092

Confusion matrix (rows=true, cols=pred):
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      0    167      2      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     28      0      0

## combining

In [53]:
good_labels = [0,1,2,3,4,5,6,11]
bad_labels = [7,8,9,10]

y_pred_final = []


for i in range(len(cat_preds)):
    # Lấy giá trị dự đoán từ cat và lstm, ép về số nguyên đơn
    cat_label = int(cat_preds[i][0]) if isinstance(cat_preds[i], np.ndarray) else int(cat_preds[i])
    lstm_label = int(lstm_preds[i][0]) if isinstance(lstm_preds[i], np.ndarray) else int(lstm_preds[i])

    # Nếu cat dự đoán vào lớp đáng tin
    if cat_label in good_labels:
        y_pred_final.append(cat_label)
    else:
        y_pred_final.append(lstm_label)

# Chuyển sang mảng numpy kiểu int
y_pred = np.array(y_pred_final).astype(int)


print("Accuracy:", accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision (macro):", precision_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("Recall (macro):",   recall_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("F1 (macro):",       f1_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
cm = confusion_matrix(y_test, y_pred, labels=labels)
print("     " + " ".join(f"{int(lbl):6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{int(lbl):3d} " + " ".join(f"{int(val):6d}" for val in row))

Accuracy: 0.8929980276134122
Precision (macro): 0.8897021447019152
Recall (macro): 0.8929980276134123
F1 (macro): 0.8899785204001803
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      1    168      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      3    164      0      2      0      0      0      0      0
  5      0      0      0      0      0    169      0      0      0      0      0      0
  6      0      0      0      2      0      0    167      0      0      0      0      0
  7      0      0     20      2      1      0      3    106     37      0      0      0
  8      0      0      0      0      0      0      0     24    145      0 

# Mutual Inference

In [71]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

def mutual_inference(gbt_probs_list, dl_probs_list, y_test,
                                   w_gbt_base=None, w_dl_base=None,
                                   alpha=0.6, beta=0.6):
    """
    Core mutual inference (no optimization)
    """
    n_gbt = len(gbt_probs_list)
    n_dl = len(dl_probs_list)
    n_samples = gbt_probs_list[0].shape[0]
    n_class = gbt_probs_list[0].shape[1]

    if w_gbt_base is None:
        w_gbt_base = np.ones(n_gbt) / n_gbt
    if w_dl_base is None:
        w_dl_base = np.ones(n_dl) / n_dl

    preds = []

    for i in range(n_samples):
        p_gbt_i = np.array([p[i] for p in gbt_probs_list])
        p_dl_i  = np.array([p[i] for p in dl_probs_list])

        c_gbt = np.array([
            np.partition(p_row, -2)[-1] - np.partition(p_row, -2)[-2]
            for p_row in p_gbt_i
        ])
        c_dl = np.array([
            1 - entropy(p_row) / np.log(n_class)
            for p_row in p_dl_i
        ])

        A = cosine_similarity(p_gbt_i, p_dl_i)
        row_mean = A.mean(axis=1)
        col_mean = A.mean(axis=0)

        w_gbt_raw = alpha * c_gbt * row_mean + (1 - alpha) * w_gbt_base
        w_dl_raw  = beta  * c_dl  * col_mean + (1 - beta)  * w_dl_base

        w_gbt = w_gbt_raw / np.sum(w_gbt_raw)
        w_dl  = w_dl_raw  / np.sum(w_dl_raw)

        p_ensemble = np.zeros(n_class)
        for k in range(n_gbt):
            p_ensemble += w_gbt[k] * p_gbt_i[k]
        for l in range(n_dl):
            p_ensemble += w_dl[l] * p_dl_i[l]

        preds.append(np.argmax(p_ensemble))

    preds = np.array(preds)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='macro', zero_division=0)
    rec = recall_score(y_test, preds, average='macro', zero_division=0)
    f1 = f1_score(y_test, preds, average='macro', zero_division=0)

    return {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}


def optimize_alpha_beta(gbt_probs_list, dl_probs_list, y_test,
                        grid_alpha=None, grid_beta=None,
                        w_gbt_base=None, w_dl_base=None):
    """
    Grid search optimization for alpha, beta to maximize F1-macro
    """
    if grid_alpha is None:
        grid_alpha = np.linspace(0.1, 0.9, 9)
    if grid_beta is None:
        grid_beta = np.linspace(0.1, 0.9, 9)

    best_f1 = -1
    best_params = None
    best_metrics = None

    print("Searching for best (alpha, beta)...")

    for a in grid_alpha:
        for b in grid_beta:
            metrics = mutual_inference(
                gbt_probs_list, dl_probs_list, y_test,
                w_gbt_base=w_gbt_base, w_dl_base=w_dl_base,
                alpha=a, beta=b
            )
            f1 = metrics['F1']
            if f1 > best_f1:
                best_f1 = f1
                best_params = (a, b)
                best_metrics = metrics

    print(f"\n✅ Best alpha={best_params[0]:.2f}, beta={best_params[1]:.2f}")
    print(f"Accuracy={best_metrics['Accuracy']:.4f}, "
          f"Precision={best_metrics['Precision']:.4f}, "
          f"Recall={best_metrics['Recall']:.4f}, "
          f"F1={best_metrics['F1']:.4f}")

    return best_params, best_metrics


gbt_probs_list = [xgb_probs, cat_probs, rf_probs, et_probs, lgbm_probs]
dl_probs_list  = [lstm_probs, dnn_probs]
# gbt_probs_list = [xgb_probs, cat_probs, et_probs, lgbm_probs]
# dl_probs_list  = [dnn_probs]

best_params, best_metrics = optimize_alpha_beta(
    gbt_probs_list, dl_probs_list, y_test
)

Searching for best (alpha, beta)...

✅ Best alpha=0.90, beta=0.10
Accuracy=0.8812, Precision=0.8799, Recall=0.8812, F1=0.8784
